# Real-time inference benchmark

Run this notebook on each device you want to compare. It measures:

- Human detector inference time
- Posture classifier inference time on crop images
- Full pipeline time: detect humans, crop each person, classify each crop

The notebook displays timing tables only. It does not save CSVs, images, or other benchmark artifacts.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import platform
import statistics

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "runs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DETECTOR_WEIGHTS = PROJECT_ROOT / "runs" / "human_detection" / "weights" / "best.pt"
CLASSIFIER_WEIGHTS = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small" / "best.pt"
CLASS_MAP = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small" / "class_to_idx.json"

DETECTION_IMAGES_DIR = PROJECT_ROOT / "data" / "labeled" / "human_detection" / "train" / "images"
CLASSIFICATION_DATA_DIR = PROJECT_ROOT / "data" / "labeled" / "posture_classification"
# Change these before running on another device if needed.
DEVICE = "auto"  # auto, cpu, cuda, mps
DETECTOR_CONFIDENCE = 0.23
POSTURE_UNKNOWN_CONFIDENCE = 0.50
DETECTOR_IMAGE_SIZE = 320
CLASSIFIER_IMAGE_SIZE = 224
CROP_PADDING = 0.12
DETECTOR_SAMPLE_IMAGES = 100
CLASSIFIER_SAMPLE_CROPS = 300
PIPELINE_SAMPLE_IMAGES = 100
WARMUP_RUNS = 5

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Project root:", PROJECT_ROOT)
print("Detector weights:", DETECTOR_WEIGHTS, DETECTOR_WEIGHTS.exists())
print("Classifier weights:", CLASSIFIER_WEIGHTS, CLASSIFIER_WEIGHTS.exists())


In [ ]:
def choose_device(requested="auto"):
    requested = requested.lower()
    if requested != "auto":
        return requested
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def sync_device(device):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif device == "mps" and hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.synchronize()
        except Exception:
            pass


def summarize_ms(values):
    values = [float(v) for v in values]
    if not values:
        return {"count": 0, "mean_ms": None, "median_ms": None, "p90_ms": None, "p95_ms": None, "min_ms": None, "max_ms": None}
    arr = np.array(values, dtype=np.float64)
    return {
        "count": len(values),
        "mean_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
        "p90_ms": float(np.percentile(arr, 90)),
        "p95_ms": float(np.percentile(arr, 95)),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
    }


def first_images(root, limit):
    images = sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    return images[: min(limit, len(images))]


def find_classification_images(root, limit):
    preferred_splits = ["test", "valid", "val", "train"]
    for split in preferred_splits:
        split_dir = root / split
        if split_dir.exists():
            images = sorted(p for p in split_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
            if images:
                return images[: min(limit, len(images))], split
    images = sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    return images[: min(limit, len(images))], "all"


def build_classifier(model_name, num_classes):
    if model_name == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=None)
    else:
        model = models.mobilenet_v3_small(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


def load_classifier(weights_path, class_map_path, device):
    class_to_idx = json.loads(class_map_path.read_text(encoding="utf-8"))
    class_names = [name for name, _idx in sorted(class_to_idx.items(), key=lambda item: item[1])]
    checkpoint = torch.load(weights_path, map_location=device, weights_only=False)
    model_name = checkpoint.get("model_name", "mobilenet_v3_small")
    model = build_classifier(model_name, len(class_names))
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    model.eval()
    return model, class_names, model_name


device = choose_device(DEVICE)
torch_device = torch.device(device)
print("Selected device:", device)
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

## Load models and sample data

In [ ]:
detector = YOLO(str(DETECTOR_WEIGHTS))
classifier, class_names, classifier_model_name = load_classifier(CLASSIFIER_WEIGHTS, CLASS_MAP, torch_device)

detector_images = first_images(DETECTION_IMAGES_DIR, max(DETECTOR_SAMPLE_IMAGES, PIPELINE_SAMPLE_IMAGES))
classification_images, classification_split = find_classification_images(CLASSIFICATION_DATA_DIR, CLASSIFIER_SAMPLE_CROPS)

classifier_transform = transforms.Compose([
    transforms.Resize((CLASSIFIER_IMAGE_SIZE, CLASSIFIER_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Detector images:", len(detector_images), DETECTION_IMAGES_DIR)
print("Classifier crops:", len(classification_images), "split:", classification_split)
print("Classifier model:", classifier_model_name, class_names)

## Detector-only benchmark

This measures one YOLO call per image. It stores wall time and Ultralytics internal speed fields when available.

In [ ]:
# Warmup
for image_path in detector_images[:WARMUP_RUNS]:
    detector.predict(source=str(image_path), imgsz=DETECTOR_IMAGE_SIZE, conf=DETECTOR_CONFIDENCE, device=device, verbose=False)
sync_device(device)

detector_rows = []
for image_path in detector_images[:DETECTOR_SAMPLE_IMAGES]:
    start = perf_counter()
    result = detector.predict(
        source=str(image_path),
        imgsz=DETECTOR_IMAGE_SIZE,
        conf=DETECTOR_CONFIDENCE,
        device=device,
        verbose=False,
    )[0]
    sync_device(device)
    total_ms = (perf_counter() - start) * 1000
    boxes = 0 if result.boxes is None else len(result.boxes)
    speed = result.speed or {}
    detector_rows.append({
        "image": str(image_path),
        "boxes": boxes,
        "total_ms": total_ms,
        "preprocess_ms": speed.get("preprocess"),
        "inference_ms": speed.get("inference"),
        "postprocess_ms": speed.get("postprocess"),
    })

detector_df = pd.DataFrame(detector_rows)
detector_summary = summarize_ms(detector_df["total_ms"])
detector_summary["stage"] = "detector_total"
detector_summary

## Classifier-only benchmark

This measures MobileNet forward time on already-cropped posture images.

In [ ]:
def classify_image_path(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = classifier_transform(image).unsqueeze(0).to(torch_device)
    with torch.inference_mode():
        probs = torch.softmax(classifier(tensor), dim=1)[0]
        top_prob, top_idx = torch.max(probs, dim=0)
    return class_names[int(top_idx.item())], float(top_prob.item())


for image_path in classification_images[:WARMUP_RUNS]:
    classify_image_path(image_path)
sync_device(device)

classifier_rows = []
for image_path in classification_images[:CLASSIFIER_SAMPLE_CROPS]:
    start = perf_counter()
    label, confidence = classify_image_path(image_path)
    sync_device(device)
    total_ms = (perf_counter() - start) * 1000
    classifier_rows.append({
        "image": str(image_path),
        "label": label,
        "confidence": confidence,
        "unknown": confidence < POSTURE_UNKNOWN_CONFIDENCE,
        "total_ms": total_ms,
    })

classifier_df = pd.DataFrame(classifier_rows)
classifier_summary = summarize_ms(classifier_df["total_ms"])
classifier_summary["stage"] = "classifier_total_per_crop"
classifier_summary

## Full pipeline benchmark

This measures the real pipeline shape: one image goes through YOLO, then every detected human crop goes through MobileNet.

In [ ]:
def ensure_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    return image


def pad_box(x1, y1, x2, y2, width, height, padding=CROP_PADDING):
    pad_x = int((x2 - x1) * padding)
    pad_y = int((y2 - y1) * padding)
    return (
        max(0, x1 - pad_x),
        max(0, y1 - pad_y),
        min(width, x2 + pad_x),
        min(height, y2 + pad_y),
    )


def classify_crop_bgr(crop):
    if crop.size == 0:
        return "unknown", 0.0
    rgb = cv2.cvtColor(ensure_bgr(crop), cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb)
    tensor = classifier_transform(pil_image).unsqueeze(0).to(torch_device)
    with torch.inference_mode():
        probs = torch.softmax(classifier(tensor), dim=1)[0]
        top_prob, top_idx = torch.max(probs, dim=0)
    confidence = float(top_prob.item())
    label = class_names[int(top_idx.item())]
    if confidence < POSTURE_UNKNOWN_CONFIDENCE:
        label = "unknown"
    return label, confidence


for image_path in detector_images[:WARMUP_RUNS]:
    image = cv2.imread(str(image_path))
    detector.predict(source=image, imgsz=DETECTOR_IMAGE_SIZE, conf=DETECTOR_CONFIDENCE, device=device, verbose=False)
sync_device(device)

pipeline_rows = []
for image_path in detector_images[:PIPELINE_SAMPLE_IMAGES]:
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    height, width = image.shape[:2]

    start_total = perf_counter()
    start_detector = perf_counter()
    result = detector.predict(
        source=image,
        imgsz=DETECTOR_IMAGE_SIZE,
        conf=DETECTOR_CONFIDENCE,
        device=device,
        verbose=False,
    )[0]
    sync_device(device)
    detector_ms = (perf_counter() - start_detector) * 1000

    labels = []
    start_classifier = perf_counter()
    if result.boxes is not None and len(result.boxes):
        xyxy = result.boxes.xyxy.detach().cpu().numpy()
        for coords in xyxy:
            x1, y1, x2, y2 = [int(round(v)) for v in coords[:4]]
            x1, y1 = max(0, min(x1, width - 1)), max(0, min(y1, height - 1))
            x2, y2 = max(0, min(x2, width - 1)), max(0, min(y2, height - 1))
            px1, py1, px2, py2 = pad_box(x1, y1, x2, y2, width, height)
            crop = image[py1:py2, px1:px2]
            labels.append(classify_crop_bgr(crop))
    sync_device(device)
    classifier_ms = (perf_counter() - start_classifier) * 1000
    total_ms = (perf_counter() - start_total) * 1000

    pipeline_rows.append({
        "image": str(image_path),
        "detected_people": len(labels),
        "detector_ms": detector_ms,
        "classifier_ms_total": classifier_ms,
        "classifier_ms_per_person": classifier_ms / len(labels) if labels else 0.0,
        "total_ms": total_ms,
        "fps_if_every_frame": 1000 / total_ms if total_ms > 0 else None,
    })

pipeline_df = pd.DataFrame(pipeline_rows)
pipeline_summary = summarize_ms(pipeline_df["total_ms"])
pipeline_summary["stage"] = "pipeline_total"
pipeline_summary

## Summary

In [ ]:
summary_rows = [detector_summary, classifier_summary, pipeline_summary]
summary_df = pd.DataFrame(summary_rows)
summary_df.insert(0, "device", device)
summary_df.insert(1, "platform", platform.platform())
summary_df.insert(2, "torch_version", torch.__version__)
summary_df.insert(3, "detector_imgsz", DETECTOR_IMAGE_SIZE)
summary_df.insert(4, "detector_confidence", DETECTOR_CONFIDENCE)
summary_df.insert(5, "posture_unknown_confidence", POSTURE_UNKNOWN_CONFIDENCE)
summary_df